In [13]:
import pandas as pd
import numpy as np
import json


In [4]:
with open('stations.json') as f:
    stations = json.load(f)

with open('trains.json') as f:
    trains = json.load(f)

with open('schedules.json') as f:
    schedules = json.load(f)

In [5]:
stations_df = pd.json_normalize(stations['features'])
trains_df = pd.DataFrame(trains)
schedules_df = pd.DataFrame(schedules)

stations_df.head()

,type,geometry.type,geometry.coordinates,properties.state,properties.code,properties.name,properties.zone,properties.address,geometry
0,Feature,Point,"[75.4516454, 27.2520587]",Rajasthan,BDHL,Badhal,NWR,"Kishangarh Renwal, Rajasthan",NaN
1,Feature,NaN,NaN,None,XX-BECE,XX-BECE,None,None,NaN
2,Feature,NaN,NaN,None,XX-BSPY,XX-BSPY,None,None,NaN
3,Feature,NaN,NaN,None,YY-BPLC,YY-BPLC,None,None,NaN
4,Feature,Point,"[79.519746, 28.913427000000002]",Uttar Pradesh,KHH,KICHHA,NER,"Kichha, Uttar Pradesh",NaN


In [6]:
print(stations_df.shape)
print(trains_df.shape)
print(schedules_df.shape)

stations_df.info()

(8990, 9)
(5208, 2)
(417080, 8)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8990 entries, 0 to 8989
Data columns (total 9 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   type                  8990 non-null   object 
 1   geometry.type         8697 non-null   object 
 2   geometry.coordinates  8697 non-null   object 
 3   properties.state      4458 non-null   object 
 4   properties.code       8990 non-null   object 
 5   properties.name       8990 non-null   object 
 6   properties.zone       4458 non-null   object 
 7   properties.address    4458 non-null   object 
 8   geometry              0 non-null      float64
dtypes: float64(1), object(8)
memory usage: 632.2+ KB


In [7]:
stations_df = stations_df[['properties.code', 'properties.name']]
stations_df.columns = ['code', 'name']
stations_df.dropna(inplace=True)

stations_df.head()

,code,name
0,BDHL,Badhal
1,XX-BECE,XX-BECE
2,XX-BSPY,XX-BSPY
3,YY-BPLC,YY-BPLC
4,KHH,KICHHA


In [8]:
schedules_df.dropna(inplace=True)

schedules_df = schedules_df[schedules_df['station_code'] != '0']

In [9]:
schedules_df['arrival'] = pd.to_datetime(schedules_df['arrival'], errors='coerce')
schedules_df['departure'] = pd.to_datetime(schedules_df['departure'], errors='coerce')

/var/folders/jq/7mz7h2f9643255w992tsk2140000gn/T/ipykernel_19424/1558200334.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  schedules_df['arrival'] = pd.to_datetime(schedules_df['arrival'], errors='coerce')
/var/folders/jq/7mz7h2f9643255w992tsk2140000gn/T/ipykernel_19424/1558200334.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  schedules_df['departure'] = pd.to_datetime(schedules_df['departure'], errors='coerce')


In [10]:
schedules_df['arrival'].head()

0   NaT
1   NaT
2   NaT
3   NaT
4   NaT
Name: arrival, dtype: datetime64[ns]

In [11]:
schedules_df['arrival'] = pd.to_datetime(schedules_df['arrival'], format='%H:%M:%S', errors='coerce')
schedules_df['departure'] = pd.to_datetime(schedules_df['departure'], format='%H:%M:%S', errors='coerce')

In [12]:
schedules_df['delay'] = (schedules_df['departure'] - schedules_df['arrival']).dt.total_seconds()

In [13]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
schedules_df['station_code'] = le.fit_transform(schedules_df['station_code'])

In [14]:
df = schedules_df.merge(trains_df, on='train_number', how='left')
df = df.merge(stations_df, left_on='station_code', right_on='code', how='left')

KeyError: 'train_number'

In [ ]:
df.drop_duplicates(inplace=True)
df.fillna(0, inplace=True)

In [15]:
df.to_csv("cleaned_railway_data.csv", index=False)

NameError: name 'df' is not defined

In [16]:
print(schedules_df.columns)
print(trains_df.columns)

Index(['arrival', 'day', 'train_name', 'station_name', 'station_code', 'id',
       'train_number', 'departure', 'delay'],
      dtype='object')
Index(['type', 'features'], dtype='object')


In [17]:
trains_df = pd.json_normalize(trains['features'])
trains_df.head()

,type,geometry.type,geometry.coordinates,properties.third_ac,properties.arrival,properties.from_station_code,properties.name,properties.zone,properties.chair_car,properties.first_class,...,properties.departure,properties.return_train,properties.to_station_code,properties.second_ac,properties.classes,properties.to_station_name,properties.duration_h,properties.type,properties.first_ac,properties.distance
0,Feature,LineString,"[[74.880117, 32.706975], [74.953339, 32.762368...",0,12:15:00,JAT,Jammu Tawi Udhampur Special,NR,0,0,...,10:40:00,04602,UHP,0,,UDHAMPUR,1.0,DEMU,0,53.0
1,Feature,LineString,"[[75.154881, 32.92664], [75.14542599999999, 32...",0,08:35:00,UHP,UDHAMPUR JAMMUTAWI DMU,NR,0,0,...,06:45:00,04601,JAT,0,,JAMMU TAWI,1.0,DEMU,0,53.0
2,Feature,LineString,"[[74.880117, 32.706975], [74.953339, 32.762368...",0,17:50:00,JAT,JAT UDAHMPUR DMU,NR,0,0,...,16:15:00,04604,UHP,0,,UDHAMPUR,1.0,DEMU,0,53.0
3,Feature,LineString,"[[75.154881, 32.92664], [75.14542599999999, 32...",0,19:50:00,UHP,UDHAMPUR JAMMUTAWI DMU,NR,0,0,...,18:20:00,04603,JAT,0,,JAMMU TAWI,1.0,DEMU,0,53.0
4,Feature,LineString,"[[72.840535, 19.061911], [72.840078, 19.069166...",1,12:30:00,BDTS,Mumbai BandraT-Bikaner SF Special,NWR,0,0,...,14:35:00,04727,BKN,1,,BIKANER JN,21.0,SF,0,1212.0


In [18]:
print(trains_df.columns)


Index(['type', 'geometry.type', 'geometry.coordinates', 'properties.third_ac',
       'properties.arrival', 'properties.from_station_code', 'properties.name',
       'properties.zone', 'properties.chair_car', 'properties.first_class',
       'properties.duration_m', 'properties.sleeper',
       'properties.from_station_name', 'properties.number',
       'properties.departure', 'properties.return_train',
       'properties.to_station_code', 'properties.second_ac',
       'properties.classes', 'properties.to_station_name',
       'properties.duration_h', 'properties.type', 'properties.first_ac',
       'properties.distance'],
      dtype='object')


In [19]:
trains_df = trains_df[['properties.number', 'properties.name']]
trains_df.columns = ['train_number', 'train_name']

In [20]:
df = schedules_df.merge(trains_df, on='train_number', how='left')

In [ ]:
df['hour'] = pd.to_datetime(df['departure']).dt.hour
df['day_of_week'] = pd.to_datetime(df['departure']).dt.dayofweek
df['is_weekend'] = df['day_of_week'].apply(lambda x: 1 if x >= 5 else 0)

In [17]:
import os
print(os.getcwd())
print(os.listdir())

df = pd.read_json('/Users/derekdsouza/Desktop/railway-ml/trains.json')

/Users/derekdsouza/Desktop/railway-ml
['trains.json', 'train-ai-ui', 'backend', 'README.md', 'Train_Throughput_AI_Project.ipynb', 'railway_preprocessing.ipynb', '.gitignore', '.ipynb_checkpoints', 'indian-railways-dataset.zip', '.git']


In [18]:
import pandas as pd

df = pd.read_json('/Users/derekdsouza/Desktop/railway-ml/trains.json')
df.head()

,type,features
0,FeatureCollection,"{'geometry': {'type': 'LineString', 'coordinat..."
1,FeatureCollection,"{'geometry': {'type': 'LineString', 'coordinat..."
2,FeatureCollection,"{'geometry': {'type': 'LineString', 'coordinat..."
3,FeatureCollection,"{'geometry': {'type': 'LineString', 'coordinat..."
4,FeatureCollection,"{'geometry': {'type': 'LineString', 'coordinat..."


In [22]:
df.columns

Index(['type', 'features'], dtype='str')

In [23]:
df['features'][0]

{'geometry': {'type': 'LineString',
  'coordinates': [[74.880117, 32.706975],
   [74.953339, 32.762368],
   [75.044078, 32.80617],
   [75.13222499999999, 32.769105],
   [75.14542599999999, 32.863528],
   [75.154881, 32.92664]]},
 'type': 'Feature',
 'properties': {'third_ac': 0,
  'arrival': '12:15:00',
  'from_station_code': 'JAT',
  'name': 'Jammu Tawi Udhampur Special',
  'zone': 'NR',
  'chair_car': 0,
  'first_class': 0,
  'duration_m': 35,
  'sleeper': 0,
  'from_station_name': 'JAMMU TAWI',
  'number': '04601',
  'departure': '10:40:00',
  'return_train': '04602',
  'to_station_code': 'UHP',
  'second_ac': 0,
  'classes': '',
  'to_station_name': 'UDHAMPUR',
  'duration_h': 1,
  'type': 'DEMU',
  'first_ac': 0,
  'distance': 53}}

In [24]:
data = [feature['properties'] for feature in df['features']]

In [25]:
df = pd.DataFrame(data)

In [26]:
df.head()

,third_ac,arrival,from_station_code,name,zone,chair_car,first_class,duration_m,sleeper,from_station_name,...,departure,return_train,to_station_code,second_ac,classes,to_station_name,duration_h,type,first_ac,distance
0,0,12:15:00,JAT,Jammu Tawi Udhampur Special,NR,0,0,35.0,0,JAMMU TAWI,...,10:40:00,04602,UHP,0,,UDHAMPUR,1.0,DEMU,0,53.0
1,0,08:35:00,UHP,UDHAMPUR JAMMUTAWI DMU,NR,0,0,50.0,0,UDHAMPUR,...,06:45:00,04601,JAT,0,,JAMMU TAWI,1.0,DEMU,0,53.0
2,0,17:50:00,JAT,JAT UDAHMPUR DMU,NR,0,0,35.0,0,JAMMU TAWI,...,16:15:00,04604,UHP,0,,UDHAMPUR,1.0,DEMU,0,53.0
3,0,19:50:00,UHP,UDHAMPUR JAMMUTAWI DMU,NR,0,0,30.0,0,UDHAMPUR,...,18:20:00,04603,JAT,0,,JAMMU TAWI,1.0,DEMU,0,53.0
4,1,12:30:00,BDTS,Mumbai BandraT-Bikaner SF Special,NWR,0,0,55.0,1,MUMBAI BANDRA TERMINUS,...,14:35:00,04727,BKN,1,,BIKANER JN,21.0,SF,0,1212.0


In [31]:
df.columns = df.columns.str.strip().str.lower()

In [32]:
print(df.columns.tolist())

['third_ac', 'arrival', 'from_station_code', 'name', 'zone', 'chair_car', 'first_class', 'duration_m', 'sleeper', 'from_station_name', 'number', 'departure', 'return_train', 'to_station_code', 'second_ac', 'classes', 'to_station_name', 'duration_h', 'type', 'first_ac', 'distance']


In [35]:
df['station_traffic'] = df.groupby('from_station_code')['number'].transform('count')

In [36]:
df[['from_station_code', 'number', 'station_traffic']].head()

,from_station_code,number,station_traffic
0,JAT,04601,37
1,UHP,04602,6
2,JAT,04603,37
3,UHP,04604,6
4,BDTS,04728,27


In [37]:
df['departure'] = pd.to_datetime(df['departure'], errors='coerce')

/var/folders/jq/7mz7h2f9643255w992tsk2140000gn/T/ipykernel_58706/1637125160.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['departure'] = pd.to_datetime(df['departure'], errors='coerce')


In [38]:
df['hour'] = df['departure'].dt.hour

In [39]:
df['day_of_week'] = df['departure'].dt.dayofweek

In [40]:
df['is_weekend'] = df['day_of_week'].apply(lambda x: 1 if x >= 5 else 0)

In [41]:
df['congestion'] = pd.qcut(df['station_traffic'], 3, labels=[0,1,2])

In [42]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['station_encoded'] = le.fit_transform(df['from_station_code'])

In [43]:
features = ['station_encoded', 'hour', 'day_of_week', 'is_weekend', 'station_traffic']
X = df[features]
y = df['congestion']

In [44]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

model = RandomForestClassifier(n_estimators=200, max_depth=10)
model.fit(X_train, y_train)

print("Accuracy:", model.score(X_test, y_test))

Accuracy: 1.0


In [45]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

model = RandomForestClassifier(n_estimators=200, max_depth=10)
model.fit(X_train, y_train)

accuracy = model.score(X_test, y_test)
print("Accuracy:", accuracy)

Accuracy: 1.0


In [46]:
'station_traffic'

'station_traffic'

In [47]:
features = ['station_encoded', 'hour', 'day_of_week', 'is_weekend']

In [59]:
X = df[features]
y = df['congestion']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42)
model.fit(X_train, y_train)

print("Accuracy:", model.score(X_test, y_test))

Accuracy: 0.6401151631477927


In [56]:
df['total_duration'] = df['duration_h'] * 60 + df['duration_m']

In [54]:
features = [
    'station_encoded',
    'hour',
    'day_of_week',
    'is_weekend',
    'distance',
    'total_duration'
]

In [57]:
le2 = LabelEncoder()
df['to_station_encoded'] = le2.fit_transform(df['to_station_code'])

In [58]:
features = [
    'station_encoded',
    'to_station_encoded',
    'hour',
    'day_of_week',
    'is_weekend',
    'distance',
    'total_duration'
]

In [60]:
model = RandomForestClassifier(
    n_estimators=300,
    max_depth=15,
    min_samples_split=3,
    min_samples_leaf=2,
    random_state=42
)

In [61]:
features = [
    'station_encoded',
    'to_station_encoded',
    'hour',
    'day_of_week',
    'is_weekend',
    'distance',
    'total_duration'
]

X = df[features]
y = df['congestion']

In [62]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [63]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=300,
    max_depth=15,
    min_samples_split=3,
    min_samples_leaf=2,
    random_state=42
)

In [64]:
model.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",300
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",15
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",3
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",2
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric(y

In [65]:
accuracy = model.score(X_test, y_test)
print("Accuracy:", accuracy)

Accuracy: 0.6746641074856046


In [66]:
from sklearn.metrics import classification_report

y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.60      0.68      0.64       334
           1       0.65      0.62      0.64       347
           2       0.78      0.72      0.75       361

    accuracy                           0.67      1042
   macro avg       0.68      0.67      0.67      1042
weighted avg       0.68      0.67      0.68      1042



In [67]:
import pickle

pickle.dump(model, open('model.pkl', 'wb'))